# Install Required Libraries

In [1]:
# Install necessary libraries
!pip install -q transformers datasets evaluate
!pip install -q pyttsx3  # For voice synthesis (text-to-speech)
!pip install -q gradio  # For creating an upload interface


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5

In [2]:
!pip install gTTS # Install the gTTS library

In [3]:
!sudo apt install espeak # Install espeak using apt package manager

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-data libespeak1 libportaudio2 libsonic0
The following NEW packages will be installed:
  espeak espeak-data libespeak1 libportaudio2 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 30 not upgraded.
Need to get 1,382 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 espeak-data amd64 1.48.15+dfsg-3 [1,085 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libespeak1 amd64 1.48.15+dfsg-3 [156 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 espeak amd64 1.48.15+dfsg-3 [64.2 kB]
Fetched 1,382 kB in 1s (992 kB

In [4]:
!gdown "1_gqdScnGPxDujbsHKiqWHyRaFhKnzCvU"
# Downloading trained DeepFake model

Downloading...
From (original): https://drive.google.com/uc?id=1_gqdScnGPxDujbsHKiqWHyRaFhKnzCvU
From (redirected): https://drive.google.com/uc?id=1_gqdScnGPxDujbsHKiqWHyRaFhKnzCvU&confirm=t&uuid=c8a67a1a-8cd8-4cc8-896e-b681f31bb6ec
To: /content/deepfake_model.zip
100% 319M/319M [00:06<00:00, 49.6MB/s]


In [5]:
# prompt: !unzip deepfake_model.zip into '/content/deepfake_model'

!unzip deepfake_model.zip -d /content/deepfake_model


Archive:  deepfake_model.zip
  inflating: /content/deepfake_model/model.safetensors  
  inflating: /content/deepfake_model/training_args.bin  
  inflating: /content/deepfake_model/config.json  


# Import Libraries and Load the Model

In [8]:
# Import Libraries and Load the Model
from transformers import ViTForImageClassification, ViTImageProcessor, pipeline
from gtts import gTTS
import gradio as gr
from PIL import Image
import numpy as np
import cv2
import os

# Define the model directory where config.json and model.safetensors are stored
model_directory = '/content/deepfake_model'

# Load the model from the safetensors file and the config.json
model = ViTForImageClassification.from_pretrained(
    model_directory,
    use_safetensors=True  # Use safetensors format for loading the model
)

# Manually create a ViTImageProcessor for the model
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# Create a prediction pipeline using the loaded model and processor
pipe = pipeline("image-classification", model=model, feature_extractor=processor)

def voice_message(prediction_label):
    if prediction_label == 'Fake':
        message = "Warning, this is a deepfake."
    else:
        message = "This image is real."

    # Generate speech using gTTS
    tts = gTTS(message)
    audio_file = "voice_message.mp3"
    tts.save(audio_file)
    return audio_file

def is_face_present(img: Image.Image) -> bool:
    # Convert PIL image to a format compatible with OpenCV
    cv_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    # Load pre-trained face detector from OpenCV
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    return len(faces) > 0

def predict_image(img: Image.Image):
    try:
        # Validate that the image is properly loaded
        if img is None:
            raise ValueError("No image provided.")

        # Check if a face is present in the image
        if not is_face_present(img):
            return "Invalid input - No face detected", None

        # Get the top prediction from the pipeline
        prediction = pipe(img)[0]
        label = prediction['label']  # Expected to be 'Real' or 'Fake'
        score = prediction['score']

        # Generate a voice message MP3 based on the prediction
        audio_file = voice_message(label)

        result_text = f"Prediction: {label}, Score: {score:.4f}"
        return result_text, audio_file

    except Exception as e:
        # Capture any exceptions (e.g., file corruption, processing errors) and return an error message
        return f"Error: {str(e)}", None


Device set to use cpu


# Gradio Interface for User to Upload an Image

In [9]:
# Create Gradio interface for image upload and prediction with two outputs: text and audio
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=[ "text", "audio" ],
    title="Deepfake Detector",
    description="Upload an image to predict if it is real or a deepfake. A voice message will be generated based on the prediction."
)

# Launch the Gradio interface
interface.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://80136ece3b1a3a7291.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://80136ece3b1a3a7291.gradio.live
